In [1]:
"""
02_predecir_test.py
Aplica el modelo final (generado por 01_entrenar_modelo_final.py) a los datos de test y escribe submission.csv.

Entrada : MODEL_DIR (modelos + features de test) y test_data.jsonlines (para validar filas)
Salida  : submission.csv  (ID desde 1, columna 'condition')
"""
import os, re
import numpy as np, pandas as pd
import joblib

MODEL_DIR = '/kaggle/input/notebooks/mirandacontieri/01-entrenar-modelo-final/modelo_final'
DATA_DIR = None                  # carpeta con test_data.jsonlines (None = buscar en /kaggle/input)
TARGET = 'condition'
OUT = 'submission.csv'

# --- modelos y features de test ---
M = joblib.load(os.path.join(MODEL_DIR, 'meta.joblib'))
meta, names, classes, N_FOLDS = M['meta'], M['names'], M['classes'], M['n_folds']
Xb = pd.read_pickle(os.path.join(MODEL_DIR, 'X_test_lgb_xgb.pkl'))
Xc_te = pd.read_pickle(os.path.join(MODEL_DIR, 'X_test_cat.pkl'))
models = {n: joblib.load(os.path.join(MODEL_DIR, f'{n}_models.joblib')) for n in names}

# --- chequeo contra el archivo de test ---
dirs = [DATA_DIR] if DATA_DIR else ['/kaggle/input', os.path.expanduser('~/.cache/kagglehub/competitions')]
paths = [os.path.join(r, f) for d in dirs if d and os.path.isdir(d) for r, _, fs in os.walk(d) for f in fs if f.endswith('.jsonlines')]
test = pd.read_json([p for p in paths if 'test' in os.path.basename(p).lower()][0], lines=True)
nte = len(test)
assert len(Xb) == nte and len(Xc_te) == nte, 'las features guardadas no coinciden con el test'

# --- predicción de cada modelo: promedio de los 5 folds, luego de las semillas ---
clean_cols = lambda d: d.set_axis([re.sub(r'[\[\]<>]', '_', str(c)) for c in d.columns], axis=1)
inputs = {'lgb': Xb, 'xgb': clean_cols(Xb), 'cat': Xc_te}
T = []
for n in names:
    ms = models[n]; per_seed = []
    for s in range(M['n_seeds'][n]):
        tp = np.zeros(nte)
        for m in ms[s * N_FOLDS:(s + 1) * N_FOLDS]: tp += m.predict_proba(inputs[n])[:, 1] / N_FOLDS
        per_seed.append(tp)
    T.append(np.mean(per_seed, axis=0))
T = np.c_[T].T

# --- mezcla con el meta-modelo ---
lg = lambda p: np.log(np.clip(p, 1e-4, 1 - 1e-4) / (1 - np.clip(p, 1e-4, 1 - 1e-4)))
t_final = meta.predict_proba(lg(T))[:, 1]

sub = pd.DataFrame({'ID': range(1, nte + 1), TARGET: [classes[i] for i in (t_final > .5).astype(int)]})
sub.to_csv(OUT, index=False)
print(sub[TARGET].value_counts(normalize=True).round(3).to_dict(), '| guardado:', OUT)

{'new': 0.528, 'used': 0.472} | guardado: submission.csv
